# BC5CDR Data Preprocessing

Description: Preprocess the BioCreative V Chemical Disease Relation (BC5CDR) corpus (URL: https://huggingface.co/datasets/bigbio/bc5cdr).

Inputs:
* Training dataset in PubTator format with 500 abstracts (TXT): CDR_TrainingSet.PubTator.txt
* Developmental dataset in PubTator format with 500 abstracts (TXT): CDR_DevelopmentalSet.PubTator.txt
* Test dataset in PubTator format with 500 abstracts (TXT): CDR_TestSet.PubTator.txt

Outputs:
* Document Ids (CSV): bc5cdr-ids.csv
* Lex/sem mapping (TSV): bc5cdr-keywords.tsv
* Preprocessed documents (JSON): bc5cdr-preprocessed.json
* Preprocessed documents formatted for use by KeyBERT and KeyLLM (JSON): bc5cdr-preprocessed-merged.json

## Preliminaries

Import required packages. Set up I/O. Load helper functions.

Import required packages.

In [1]:
import re
import csv
import unicodedata
import inflect
import json
#import numpy as np
#import bs4
#from bs4 import BeautifulSoup
#import pandas as pd

## Helper Functions

In [2]:
# ------------------------------------------------------------
# Lexical unit normalization
# ------------------------------------------------------------

def normalize_annotation(term):
    """
    Convert an annotated term to a single _lex token.

    Examples:
        Tricuspid valve regurgitation
            -> tricuspid_valve_regurgitation_lex

        lithium carbonate
            -> lithium_carbonate_lex

        neurologically-impaired
            -> neurologically_impaired_lex
    """

    # Apply essentially the same basic normalization
    words = normalize(term.split())

    # Join the words into one lexical token
    term = "_".join(words)

    return term + "_lex"

# ------------------------------------------------------------
# Normalization functions
# ------------------------------------------------------------

def remove_non_ascii(words):
    """Remove non-ASCII characters from list of tokenized words"""
    new_words = []
    for word in words:
        new_word = unicodedata.normalize('NFKD', word).encode('ascii', 'ignore').decode('utf-8', 'ignore')
        new_words.append(new_word)
    return new_words

def to_lowercase(words):
    """Convert all characters to lowercase from list of tokenized words"""
    new_words = []
    for word in words:
        new_word = word.lower()
        new_words.append(new_word)
    return new_words

def remove_punctuation(words):
    """Remove punctuation from list of tokenized words"""
    new_words = []
    for word in words:
        new_word = re.sub(r'[^\w\s]', '', word)
        if new_word != '':
            new_words.append(new_word)
    return new_words

def replace_numbers(words):
    """Replace all interger occurrences in list of tokenized words with textual representation"""
    p = inflect.engine()
    new_words = []
    for word in words:
        if word.isdigit():
            new_word = p.number_to_words(word)
            new_words.append(new_word)
        else:
            new_words.append(word)
    return new_words

def normalize(words):
    words = remove_non_ascii(words)
    words = to_lowercase(words)
    words = remove_punctuation(words)
    words = replace_numbers(words)
    return words

# ------------------------------------------------------------
# Process an individual article
# ------------------------------------------------------------

def preprocess_article(title, abstract, annotations):
    """
    Preprocess one PubTator article.

    Annotation offsets in PubTator are relative to:

        title + "\\n" + abstract

    Annotated spans become single _lex tokens.
    Everything else is processed with normalize().
    """

    # PubTator offsets are based on title + newline + abstract
    text = title + "\n" + abstract

    # Sort annotations by character position
    annotations = sorted(
        annotations,
        key=lambda x: (x["start"], x["end"])
    )

    output_tokens = []
    position = 0

    for annotation in annotations:

        start = annotation["start"]
        end = annotation["end"]
        mention = annotation["mention"]

        # ----------------------------------------------------
        # Process ordinary text before this annotation
        # ----------------------------------------------------
        if start > position:
            ordinary_text = text[position:start]

            ordinary_words = ordinary_text.split()
            ordinary_words = normalize(ordinary_words)

            output_tokens.extend(ordinary_words)

        # ----------------------------------------------------
        # Add annotation as one _lex token
        # ----------------------------------------------------
        lex_term = normalize_annotation(mention)
        output_tokens.append(lex_term)

        position = end

    # --------------------------------------------------------
    # Process remaining ordinary text
    # --------------------------------------------------------
    if position < len(text):
        ordinary_text = text[position:]

        ordinary_words = ordinary_text.split()
        ordinary_words = normalize(ordinary_words)

        output_tokens.extend(ordinary_words)

    # Guarantee exactly one space between tokens
    return " ".join(output_tokens)


# ------------------------------------------------------------
# Read PubTator file
# ------------------------------------------------------------

def read_pubtator(filenames):
    """
    Read a BC5CDR/PubTator file and return preprocessed articles.
    """

    articles = []

    current_title = None
    current_abstract = None
    current_annotations = []

    for filename in filenames:
        with open(filename, "r", encoding="utf-8") as f:
    
            for line in f:
                line = line.rstrip("\n")
    
                # ------------------------------------------------
                # Blank line marks the end of an article
                # ------------------------------------------------
                if line.strip() == "":
    
                    if current_title is not None:
                        article = preprocess_article(
                            current_title,
                            current_abstract,
                            current_annotations
                        )
    
                        articles.append(article)
    
                    current_title = None
                    current_abstract = None
                    current_annotations = []
    
                    continue
    
                # ------------------------------------------------
                # Title
                # ------------------------------------------------
                if "|t|" in line:
                    pmid, current_title = line.split("|t|", 1)
    
                # ------------------------------------------------
                # Abstract
                # ------------------------------------------------
                elif "|a|" in line:
                    pmid, current_abstract = line.split("|a|", 1)
    
                # ------------------------------------------------
                # Entity annotation
                # ------------------------------------------------
                else:
                    fields = line.split("\t")
    
                    # Entity annotation lines have six fields.
                    # CID relation lines have four and are ignored.
                    if len(fields) == 6:
    
                        pmid, start, end, mention, entity_class, identifier = fields
    
                        current_annotations.append({
                            "start": int(start),
                            "end": int(end),
                            "mention": mention,
                            "class": entity_class,
                            "identifier": identifier
                        })
    
        # --------------------------------------------------------
        # Handle final article if file does not end in blank line
        # --------------------------------------------------------
        if current_title is not None:
            article = preprocess_article(
                current_title,
                current_abstract,
                current_annotations
            )
    
            articles.append(article)

    return articles

## Lexical Unit Annotation

In [3]:
# ------------------------------------------------------------
# Create table of lexical units and their domains
# ------------------------------------------------------------

input_files = [
    "../0-data-raw/CDR_TrainingSet.PubTator.txt",
    "../0-data-raw/CDR_DevelopmentSet.PubTator.txt",
    "../0-data-raw/CDR_TestSet.PubTator.txt"
]
output_file = "bc5cdr-keywords.tsv"

terms = {}

for input_file in input_files:
    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
    
            # Entity annotation lines have six tab-separated fields
            fields = line.split("\t")
    
            if len(fields) != 6:
                continue
    
            pmid, start, end, mention, entity_class, identifier = fields
    
            # Keep only Chemical and Disease annotations
            if entity_class not in {"Chemical", "Disease"}:
                continue
    
            term = normalize_annotation(mention)
    
            # Remove duplicate terms
            if term not in terms:
                terms[term] = entity_class


with open(output_file, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="\t")

    writer.writerow(["lex", "sem"])

    for term, entity_class in terms.items():
        writer.writerow([term, entity_class])

print(f"Wrote {len(terms)} unique lexical units to {output_file}")

Wrote 4871 unique lexical units to bc5cdr-keywords.tsv


## Preprocess Abstracts for Analysis

In [4]:
# ------------------------------------------------------------
# Preprocess articles and write to JSON
# ------------------------------------------------------------

input_files = [
    "../0-data-raw/CDR_TrainingSet.PubTator.txt",
    "../0-data-raw/CDR_DevelopmentSet.PubTator.txt",
    "../0-data-raw/CDR_TestSet.PubTator.txt"
]
output_file = "bc5cdr-preprocessed.json"

articles = read_pubtator(input_files)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        articles,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Wrote {len(articles)} articles to {output_file}")

Wrote 1500 articles to bc5cdr-preprocessed.json


## Preprocess for KeyBERT and KeyLLM

In [6]:
# ------------------------------------------------------------
# Create version with _lex postfixes removed
# ------------------------------------------------------------

articles_no_lex = [
    article.replace("_lex", "")
    for article in articles
]

output_file_no_lex = "bc5cdr-preprocessed-no-lex.json"

with open(output_file_no_lex, "w", encoding="utf-8") as f:
    json.dump(
        articles_no_lex,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    f"Wrote {len(articles_no_lex)} articles "
    f"to {output_file_no_lex}"
)

Wrote 1500 articles to bc5cdr-preprocessed-no-lex.json
